# 예제 03. 검증과 되먹임 — `ask_json()` 완성

전산실험 2 · AI Agent 만들기 · **3주차** · 예상 소요 45분

## 목표
- 스키마로 **내용**을 검증한다 (파싱과 다른 층위)
- 도메인 지식을 검증 규칙으로 코드에 심는다
- 실패를 모델에게 **되먹여** 스스로 고치게 한다
- 단계별 성공률을 표로 정리한다 (ablation)

> **오늘의 핵심.** 여기서 만드는 되먹임 구조가
> 6주차 ReAct 루프와 7주차 코드 수정 루프의 원형입니다.

### 준비 — 공용 모듈과 데이터 만들기 / Setup — write the shared modules and data

이 수업 저장소에는 노트북(`.ipynb`)만 둡니다. 그래서 실습에 필요한 공용 모듈과
데이터를 **아래 셀들이 직접 파일로 만들어 냅니다.** 내려받지 않으므로 네트워크가
느려도, 저장소가 비공개로 바뀌어도 그대로 동작합니다.

아래 셀을 **위에서부터 차례로 한 번씩** 실행한 뒤 다음으로 넘어가세요.
내용을 지금 읽을 필요는 없습니다. 만들어진 `.py` 파일은 왼쪽 파일 탐색기에서
언제든 열어 볼 수 있습니다.

The course repository contains notebooks only. The cells below **write the shared
modules and data files themselves** instead of downloading them. Run them once,
top to bottom, then continue. You do not need to read them now — the generated
`.py` files can be opened from the file browser on the left.

In [ ]:
%%writefile kmu_llm.py
"""전산실험2 - AI Agent 만들기 : 공용 LLM 래퍼.

계명대 통계학과 서버와 Google Gemini를 같은 방식으로 호출한다.
둘 다 OpenAI 호환 API를 제공하므로 openai SDK 하나로 통일하고
base_url 과 model 만 바꾼다.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 이 저장소는 공개(public)이므로 **학과 서버 주소와 API 키를 코드에 적지
 않는다.** 둘 다 수업시간에 공개하고, 각자 Colab Secrets 에 저장해서 쓴다.

     KMU_BASE_URL  = 수업시간에 공개  (예: https://.../v1)
     KMU_API_KEY   = 수업시간에 공개

 왼쪽 열쇠 아이콘(🔑) → 새 secret → 노트북 액세스 토글 켜기
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

사용 예:
    from kmu_llm import LLM, ask, check_server

    check_server()                       # 서버가 살아 있는지 먼저 확인
    print(ask("표준편차를 한 문장으로 설명해줘"))

    llm = LLM(provider="gemini")         # 백업 경로
    print(llm.chat([{"role": "user", "content": "안녕"}]))
"""

from __future__ import annotations

import os
import time
from typing import Any, Iterator

__all__ = [
    "KMU_MODEL",
    "GEMINI_BASE_URL",
    "GEMINI_MODEL",
    "PROVIDERS",
    "LLM",
    "ask",
    "get_secret",
    "get_key",
    "get_base_url",
    "check_server",
]

# --------------------------------------------------------------------------
# 서버 설정
# --------------------------------------------------------------------------
# 학과 서버 주소는 여기에 적지 않는다. Colab Secrets 의 KMU_BASE_URL 에서 읽는다.
# 모델 태그는 공개해도 무방하므로 상수로 둔다.
KMU_MODEL = os.environ.get("KMU_MODEL", "gemma4")

# 백업 경로. Google AI Studio 에서 무료 키를 발급받아 쓴다. (공개 주소)
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GEMINI_MODEL = "gemini-2.0-flash"

PROVIDERS: dict[str, dict[str, Any]] = {
    "kmu": {
        "base_url": None,  # 실행 시점에 Secrets 에서 읽는다
        "base_url_key": "KMU_BASE_URL",
        "model": KMU_MODEL,
        "key_name": "KMU_API_KEY",
        "label": "계명대 학과 서버",
    },
    "gemini": {
        "base_url": GEMINI_BASE_URL,
        "base_url_key": None,
        "model": GEMINI_MODEL,
        "key_name": "GEMINI_API_KEY",
        "label": "Google Gemini (백업)",
    },
}


# --------------------------------------------------------------------------
# 비밀정보 가져오기
# --------------------------------------------------------------------------
def get_secret(name: str, *, prompt: bool = True, secret: bool = True) -> str:
    """설정값을 안전한 순서로 찾는다.

    1) Colab Secrets (왼쪽 열쇠 아이콘)  2) 환경변수  3) 직접 입력

    값을 코드에 적지 않는 것이 핵심이다. 노트북이나 모듈을 GitHub 에 올리는
    순간 코드에 적힌 값은 전 세계에 공개된다. API 키뿐 아니라 **내부 서버
    주소도** 마찬가지다.

    Parameters
    ----------
    secret : True 면 입력을 화면에 표시하지 않는다(getpass).
             주소처럼 오타 확인이 중요한 값은 False 로 준다.
    """
    # 1) Colab Secrets
    try:
        from google.colab import userdata  # type: ignore

        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass

    # 2) 환경변수
    value = os.environ.get(name)
    if value:
        return value.strip()

    # 3) 직접 입력
    if not prompt:
        raise RuntimeError(
            f"'{name}' 를 찾을 수 없습니다.\n"
            f"  Colab 왼쪽 열쇠 아이콘(🔑) → 새 secret → 이름 '{name}'\n"
            f"  값은 수업시간에 공개합니다. 노트북 액세스 토글도 켜세요."
        )

    print(f"[{name}] 를 찾지 못했습니다. 직접 입력해 주세요.")
    print(f"   (Colab Secrets 에 '{name}' 로 저장해 두면 다시 묻지 않습니다)")

    if secret:
        import getpass

        value = getpass.getpass(f"{name}: ").strip()
    else:
        value = input(f"{name}: ").strip()

    if not value:
        raise RuntimeError(f"'{name}' 에 빈 값이 입력되었습니다.")
    os.environ[name] = value  # 같은 런타임 안에서는 다시 묻지 않는다
    return value


def get_key(key_name: str = "KMU_API_KEY", *, prompt: bool = True) -> str:
    """API 키를 가져온다. (get_secret 의 별칭 — 1주차 이름 유지)"""
    return get_secret(key_name, prompt=prompt, secret=True)


def get_base_url(provider: str = "kmu", *, prompt: bool = True) -> str:
    """서버 주소를 가져온다. 끝의 '/' 는 떼고 돌려준다.

    학과 서버 주소는 저장소에 적지 않으므로 Secrets 에서 읽는다.
    '/chat/completions' 까지 붙여서 저장한 경우도 알아서 잘라낸다.
    """
    conf = PROVIDERS[provider]

    if conf["base_url"]:
        return str(conf["base_url"]).rstrip("/")

    url = get_secret(conf["base_url_key"], prompt=prompt, secret=False).rstrip("/")

    # openai SDK 는 base_url 뒤에 /chat/completions 를 스스로 붙인다.
    # 주소를 통째로 저장한 경우를 대비해 정리해 준다.
    for suffix in ("/chat/completions", "/completions"):
        if url.endswith(suffix):
            url = url[: -len(suffix)]

    conf["base_url"] = url  # 같은 세션에서는 재사용
    return url


# --------------------------------------------------------------------------
# LLM 클라이언트
# --------------------------------------------------------------------------
class LLM:
    """OpenAI 호환 서버를 감싼 얇은 클라이언트.

    Parameters
    ----------
    provider : "kmu" | "gemini"
        어느 서버를 쓸지. 기본은 학과 서버.
    model, base_url, api_key :
        직접 지정하면 기본값을 덮어쓴다. (보통 지정하지 않는다)
    """

    def __init__(
        self,
        provider: str = "kmu",
        model: str | None = None,
        base_url: str | None = None,
        api_key: str | None = None,
        timeout: float = 120.0,
    ) -> None:
        if provider not in PROVIDERS:
            raise ValueError(
                f"알 수 없는 provider: {provider!r}. 가능한 값: {list(PROVIDERS)}"
            )
        conf = PROVIDERS[provider]

        self.provider = provider
        self.label = conf["label"]
        self.model = model or conf["model"]
        self.base_url = base_url or get_base_url(provider)
        self.api_key = api_key or get_key(conf["key_name"])

        try:
            from openai import OpenAI
        except ImportError as exc:  # pragma: no cover
            raise ImportError(
                "openai 패키지가 필요합니다.  !pip install -q openai"
            ) from exc

        self.client = OpenAI(
            api_key=self.api_key, base_url=self.base_url, timeout=timeout
        )

    def __repr__(self) -> str:
        # 주소는 출력하지 않는다. 화면 공유·스크린샷으로 새어나가기 쉽다.
        return f"LLM(provider={self.provider!r}, model={self.model!r})"

    # -- 기본 호출 --------------------------------------------------------
    def raw(self, messages: list[dict[str, str]], **kw: Any):
        """응답 객체 전체를 그대로 돌려준다. usage(토큰 수) 확인용."""
        params: dict[str, Any] = {
            "model": self.model,
            "messages": messages,
            "temperature": 0.7,
            "max_tokens": 1024,
        }
        params.update(kw)
        return self.client.chat.completions.create(**params)

    def chat(self, messages: list[dict[str, str]], **kw: Any) -> str:
        """대화 메시지를 보내고 응답 '텍스트'만 돌려준다."""
        response = self.raw(messages, **kw)
        content = response.choices[0].message.content
        return (content or "").strip()

    def stream(self, messages: list[dict[str, str]], **kw: Any) -> Iterator[str]:
        """응답을 조각(chunk) 단위로 흘려보낸다. 타이핑되는 효과."""
        kw["stream"] = True
        for chunk in self.raw(messages, **kw):
            if not chunk.choices:
                continue
            piece = chunk.choices[0].delta.content
            if piece:
                yield piece

    # -- 편의 메서드 ------------------------------------------------------
    def ask(self, prompt: str, system: str | None = None, **kw: Any) -> str:
        """한 번짜리 질문. messages 를 매번 만들기 번거로울 때 쓴다."""
        messages: list[dict[str, str]] = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        return self.chat(messages, **kw)

    def ask_retry(
        self,
        prompt: str,
        system: str | None = None,
        *,
        n_retry: int = 3,
        wait: float = 2.0,
        **kw: Any,
    ) -> str:
        """일시적인 네트워크·서버 오류를 재시도한다.

        30회 반복 실험처럼 호출이 많은 실습에서 한 번의 실패로 전체가
        멈추지 않도록 하기 위한 것이다.
        """
        last_error: Exception | None = None
        for attempt in range(1, n_retry + 1):
            try:
                return self.ask(prompt, system, **kw)
            except Exception as exc:  # 네트워크·서버 오류 전반
                last_error = exc
                if attempt < n_retry:
                    print(f"   [재시도 {attempt}/{n_retry - 1}] {type(exc).__name__}")
                    time.sleep(wait * attempt)  # 점점 더 오래 기다린다
        raise RuntimeError(f"{n_retry}회 시도 모두 실패했습니다.") from last_error

    def count_tokens(self, messages: list[dict[str, str]]) -> int:
        """입력 토큰 수를 서버에 물어본다. 출력은 1토큰으로 잘라 낭비를 줄인다."""
        return self.raw(messages, max_tokens=1).usage.prompt_tokens


# --------------------------------------------------------------------------
# 모듈 수준 편의 함수
# --------------------------------------------------------------------------
_default_llm: LLM | None = None


def ask(
    prompt: str, system: str | None = None, *, provider: str = "kmu", **kw: Any
) -> str:
    """가장 간단한 한 줄 호출.  ask("안녕") -> "안녕하세요..."

    내부적으로 LLM 객체를 한 번만 만들어 재사용한다.
    """
    global _default_llm
    if _default_llm is None or _default_llm.provider != provider:
        _default_llm = LLM(provider=provider)
    return _default_llm.ask(prompt, system, **kw)


# --------------------------------------------------------------------------
# 서버 진단
# --------------------------------------------------------------------------
def check_server(provider: str = "kmu", base_url: str | None = None) -> dict[str, Any]:
    """서버가 살아 있는지, 어떤 모델이 올라가 있는지 확인한다.

    수업 첫 실습. 실패하면 무엇을 확인해야 하는지 함께 출력한다.

    Returns
    -------
    dict : {"ok": bool, "models": list[str], "error": str | None}
    """
    import requests

    conf = PROVIDERS[provider]
    base_url = (base_url or get_base_url(provider)).rstrip("/")
    api_key = get_key(conf["key_name"])

    # 주소 전체를 출력하지 않는다. 호스트만 가려서 보여준다.
    print(f"조회 중: GET {_mask_url(base_url)}/models")

    try:
        response = requests.get(
            f"{base_url}/models",
            headers={"Authorization": f"Bearer {api_key}"},
            timeout=15,
        )
    except Exception as exc:
        print(f"\n[실패] 서버에 연결하지 못했습니다: {type(exc).__name__}")
        _print_troubleshooting()
        return {"ok": False, "models": [], "error": str(exc)}

    if response.status_code != 200:
        print(f"\n[실패] HTTP {response.status_code}")
        print(f"   응답: {response.text[:300]}")
        if response.status_code in (401, 403):
            print("   -> API 키가 잘못되었거나 만료되었을 가능성이 높습니다.")
        elif response.status_code == 404:
            print("   -> KMU_BASE_URL 이 '/v1' 로 끝나는지 확인하세요.")
        return {"ok": False, "models": [], "error": f"HTTP {response.status_code}"}

    models = [m.get("id", "?") for m in response.json().get("data", [])]

    print(f"\n[성공] 서버가 응답했습니다. 사용 가능한 모델 {len(models)}개:")
    for name in models:
        mark = "  <- 기본 모델" if name == conf["model"] else ""
        print(f"   - {name}{mark}")

    if conf["model"] not in models and models:
        print(
            f"\n[주의] 설정된 모델 '{conf['model']}' 이 목록에 없습니다.\n"
            f"       위 목록에서 골라 LLM(model='...') 로 지정하거나\n"
            f"       환경변수 KMU_MODEL 을 설정하세요."
        )

    return {"ok": True, "models": models, "error": None}


def _mask_url(url: str) -> str:
    """화면 공유·스크린샷 유출을 막기 위해 호스트를 가린다."""
    import re

    return re.sub(r"//([^/]+)", lambda m: "//" + m.group(1)[:4] + "***", url, count=1)


def _print_troubleshooting() -> None:
    """연결 실패 시 확인할 것들."""
    print("\n확인해 볼 것:")
    print("   1. Colab Secrets 의 KMU_BASE_URL 값이 정확합니까?")
    print("      - http / https 를 맞게 적었습니까?")
    print("      - 포트가 필요할 수 있습니다.")
    print("      - 끝이 '/v1' 이어야 합니다. ('/chat/completions' 는 빼도 됩니다)")
    print("   2. 학과 서버가 켜져 있습니까? 교내망에서만 열려 있지 않습니까?")
    print("   3. 다른 주소를 시험해 보려면:")
    print("      check_server(base_url='...')")
    print("   4. 계속 안 되면 백업 경로를 쓰세요:  LLM(provider='gemini')")


In [ ]:
%%writefile structured.py
"""전산실험2 3주차 : 구조화 출력 — 파싱 · 검증 · 재시도.

소형 모델은 "JSON으로만 답하라"고 지시해도 자주 어긴다.
코드펜스로 감싸거나, 인사를 덧붙이거나, 따옴표를 빠뜨린다.

이 모듈은 세 겹의 방어를 제공한다.

    1) 프롬프트로 형식을 강제하고        (build_json_prompt)
    2) 지저분한 응답에서 JSON을 긁어내고  (extract_json)
    3) 스키마로 검증한 뒤, 실패하면       (validate)
       오류를 되먹여 다시 요구한다        (ask_json)

사용 예:
    from kmu_llm import LLM
    from structured import ask_json

    schema = {
        "required": ["test", "reason"],
        "types": {"test": str, "reason": str},
        "allowed": {"test": ["t-test", "anova", "chi-square"]},
    }
    result = ask_json("두 그룹의 평균 비교에 쓸 검정은?", schema=schema)
    print(result["test"])
"""

from __future__ import annotations

import json
import re
from typing import Any

__all__ = [
    "JSONParseError",
    "SchemaError",
    "extract_json",
    "parse_json",
    "validate",
    "build_json_prompt",
    "ask_json",
]


class JSONParseError(ValueError):
    """모델 응답에서 JSON을 뽑아내지 못했을 때."""


class SchemaError(ValueError):
    """JSON은 얻었지만 요구한 구조가 아닐 때."""


# --------------------------------------------------------------------------
# 1) 지저분한 응답에서 JSON 긁어내기
# --------------------------------------------------------------------------
_FENCE = re.compile(r"```(?:json|python)?\s*(.*?)```", re.DOTALL)


def extract_json(text: str) -> str:
    """모델 응답에서 JSON 문자열만 뽑아낸다.

    소형 모델이 실제로 내놓는 형태들:
        ```json\\n{...}\\n```
        네, 알겠습니다! {...}
        {...}\\n도움이 되었길 바랍니다.

    Returns
    -------
    str : JSON 으로 보이는 부분

    Raises
    ------
    JSONParseError : 중괄호/대괄호 쌍을 찾지 못한 경우
    """
    if not text or not text.strip():
        raise JSONParseError("응답이 비어 있습니다.")

    # (1) 코드펜스가 있으면 그 안쪽을 우선 사용
    fence = _FENCE.search(text)
    if fence:
        text = fence.group(1)

    # (2) 가장 바깥쪽 { } 또는 [ ] 를 찾는다
    for opener, closer in (("{", "}"), ("[", "]")):
        start = text.find(opener)
        end = text.rfind(closer)
        if start != -1 and end > start:
            return text[start : end + 1].strip()

    raise JSONParseError(
        f"JSON 구조를 찾지 못했습니다. 응답 앞부분: {text[:100]!r}"
    )


def parse_json(text: str) -> Any:
    """extract_json + json.loads. 실패하면 JSONParseError."""
    snippet = extract_json(text)
    try:
        return json.loads(snippet)
    except json.JSONDecodeError as exc:
        # 흔한 사고: 마지막 항목 뒤의 쉼표(trailing comma)
        repaired = re.sub(r",\s*([}\]])", r"\1", snippet)
        try:
            return json.loads(repaired)
        except json.JSONDecodeError:
            pass
        raise JSONParseError(
            f"JSON 파싱 실패 ({exc.msg}, {exc.lineno}행 {exc.colno}열). "
            f"문제 부분: {snippet[max(0, exc.pos - 30) : exc.pos + 30]!r}"
        ) from exc


# --------------------------------------------------------------------------
# 2) 스키마 검증
# --------------------------------------------------------------------------
def validate(data: Any, schema: dict | None) -> list[str]:
    """아주 단순한 스키마 검증기. 문제점 목록을 돌려준다 (빈 목록 = 통과).

    스키마 형식::

        {
            "required": ["test", "reason"],          # 반드시 있어야 할 키
            "types":    {"alpha": float, "n": int},  # 값의 파이썬 타입
            "allowed":  {"test": ["t-test", ...]},   # 허용되는 값 목록
            "range":    {"alpha": (0.0, 1.0)},       # 수치 범위 (닫힌 구간)
        }

    pydantic 같은 라이브러리를 쓸 수도 있지만, 여기서는 검증이 무엇을 하는
    일인지 보이도록 직접 구현한다.
    """
    if schema is None:
        return []

    problems: list[str] = []

    if not isinstance(data, dict):
        return [f"최상위가 객체(dict)가 아니라 {type(data).__name__} 입니다."]

    for key in schema.get("required", []):
        if key not in data:
            problems.append(f"필수 키 '{key}' 가 없습니다.")

    for key, expected in schema.get("types", {}).items():
        if key not in data:
            continue
        value = data[key]
        # int 는 float 자리에 들어와도 허용한다
        if expected is float and isinstance(value, int) and not isinstance(value, bool):
            continue
        if not isinstance(value, expected):
            problems.append(
                f"'{key}' 는 {expected.__name__} 여야 하는데 "
                f"{type(value).__name__} 입니다 (값: {value!r})."
            )

    for key, options in schema.get("allowed", {}).items():
        if key in data and data[key] not in options:
            problems.append(
                f"'{key}' 의 값 {data[key]!r} 는 허용 목록에 없습니다. "
                f"가능한 값: {options}"
            )

    for key, (low, high) in schema.get("range", {}).items():
        if key not in data:
            continue
        value = data[key]
        if not isinstance(value, (int, float)) or isinstance(value, bool):
            problems.append(f"'{key}' 가 숫자가 아니어서 범위를 확인할 수 없습니다.")
        elif not (low <= value <= high):
            problems.append(f"'{key}' 의 값 {value} 가 [{low}, {high}] 범위 밖입니다.")

    return problems


# --------------------------------------------------------------------------
# 3) 프롬프트로 형식 강제하기
# --------------------------------------------------------------------------
def build_json_prompt(prompt: str, schema: dict | None = None, example: Any = None) -> str:
    """JSON 출력을 유도하는 지시문을 프롬프트에 덧붙인다.

    소형 모델에서 실제로 효과가 있었던 순서대로 넣는다.
    1) 출력 예시  2) 금지 사항 명시  3) 시작·끝 문자 지정
    """
    parts = [prompt, "", "## 출력 형식"]

    if example is not None:
        parts.append("아래와 정확히 같은 형식의 JSON으로만 답하세요.")
        parts.append(json.dumps(example, ensure_ascii=False, indent=2))
    elif schema and schema.get("required"):
        keys = ", ".join(f'"{k}"' for k in schema["required"])
        parts.append(f"다음 키를 가진 JSON 객체로만 답하세요: {keys}")

    if schema:
        for key, options in schema.get("allowed", {}).items():
            parts.append(f'- "{key}" 는 다음 중 하나여야 합니다: {options}')

    parts += [
        "",
        "규칙:",
        "- 설명, 인사, 사족을 쓰지 마세요.",
        "- 코드블록 표시(백틱)를 쓰지 마세요.",
        "- 응답은 반드시 { 로 시작해서 } 로 끝나야 합니다.",
    ]
    return "\n".join(parts)


# --------------------------------------------------------------------------
# 4) 파싱 · 검증 · 재시도를 합친 것
# --------------------------------------------------------------------------
def ask_json(
    prompt: str,
    system: str | None = None,
    *,
    schema: dict | None = None,
    example: Any = None,
    n_retry: int = 3,
    llm: Any = None,
    verbose: bool = False,
    **kw: Any,
) -> Any:
    """모델에게 JSON을 받아 파싱·검증한 결과를 돌려준다.

    실패하면 **오류 메시지를 모델에게 되돌려주고 다시 요구한다.**
    이 되먹임 구조가 6주차 ReAct 루프의 핵심 아이디어와 같다.

    Parameters
    ----------
    schema  : validate() 형식의 스키마
    example : 출력 예시 (few-shot). 소형 모델에서 가장 효과가 크다
    n_retry : 최대 시도 횟수
    llm     : LLM 인스턴스. 없으면 kmu_llm.LLM() 을 만든다
    verbose : 시도 과정을 출력

    Raises
    ------
    JSONParseError / SchemaError : n_retry 회 모두 실패한 경우
    """
    if llm is None:
        from kmu_llm import LLM

        llm = LLM()

    kw.setdefault("temperature", 0.0)  # 형식을 지켜야 하므로 낮게

    messages: list[dict[str, str]] = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": build_json_prompt(prompt, schema, example)})

    last_error: Exception | None = None

    for attempt in range(1, n_retry + 1):
        raw = llm.chat(messages, **kw)
        if verbose:
            print(f"[시도 {attempt}] 원문: {raw[:120]!r}")

        try:
            data = parse_json(raw)
        except JSONParseError as exc:
            last_error = exc
            feedback = (
                f"방금 응답을 JSON으로 읽을 수 없었습니다. 오류: {exc}\n"
                "설명 없이 JSON 객체만 다시 출력하세요."
            )
        else:
            problems = validate(data, schema)
            if not problems:
                if verbose:
                    print(f"[시도 {attempt}] 성공")
                return data
            last_error = SchemaError("; ".join(problems))
            feedback = (
                "JSON 형식은 맞지만 내용에 문제가 있습니다:\n"
                + "\n".join(f"- {p}" for p in problems)
                + "\n이 문제들을 고쳐서 JSON 객체만 다시 출력하세요."
            )

        if verbose:
            print(f"[시도 {attempt}] 실패 -> 되먹임: {last_error}")

        # 실패한 응답과 오류를 대화에 남겨서 모델이 스스로 고치게 한다
        messages.append({"role": "assistant", "content": raw})
        messages.append({"role": "user", "content": feedback})

    raise type(last_error)(  # type: ignore[misc]
        f"{n_retry}회 시도 모두 실패했습니다. 마지막 오류: {last_error}"
    )


In [ ]:
!pip install -q openai

# 이 수업 저장소에서 공용 모듈 내려받기

from kmu_llm import LLM, ask, check_server

PROVIDER = "kmu"          # 학과 서버가 안 되면 "gemini" 로 바꾸세요
llm = LLM(provider=PROVIDER)
print("준비 완료:", llm)

---
## 1. 검증 — 파싱과는 다른 층위

```
파싱 :  이 문자열이 JSON 문법에 맞는가?        (문법)
검증 :  이 JSON을 우리 코드가 쓸 수 있는가?     (의미)
```

검증 규칙은 네 가지면 대부분 커버됩니다.

| 규칙 | 무엇을 막나 |
|---|---|
| `required` | 키 누락 → `KeyError` |
| `types` | 타입 불일치 → 계산 오류 |
| `allowed` | 존재하지 않는 검정 이름 |
| `range` | `alpha=5` 같은 통계적 난센스 |

In [ ]:
from structured import validate

검정_스키마 = {
    "required": ["test", "alpha", "reason"],
    "types":    {"test": str, "alpha": float, "reason": str},
    "allowed":  {"test": ["independent-t-test", "paired-t-test",
                          "mann-whitney-u", "one-way-anova",
                          "chi-square", "pearson-correlation"]},
    "range":    {"alpha": (0.0, 0.2)},
}

사례들 = [
    {"test": "independent-t-test", "alpha": 0.05, "reason": "두 독립 표본의 평균 비교"},
    {"test": "티검정", "alpha": 0.05, "reason": "..."},
    {"test": "independent-t-test", "alpha": 5, "reason": "..."},
    {"test": "independent-t-test", "alpha": 0.05},
]

for d in 사례들:
    문제 = validate(d, 검정_스키마)
    상태 = "통과" if not 문제 else "실패"
    print(f"[{상태}] {d}")
    for p in 문제:
        print(f"        - {p}")
    print()

> ### 통계 전공자가 여기서 할 일
>
> `"range": {"alpha": (0.0, 0.2)}` 이 한 줄을 보십시오.
> `alpha=5` 는 문법적으로 완벽한 JSON이지만 통계적으로는 말이 안 됩니다.
>
> **모델에게 "유의수준은 0과 1 사이여야 해"라고 부탁하지 마십시오.**
> 코드로 강제하십시오. 도메인 지식을 검증 규칙으로 옮기는 것이
> 통계 에이전트를 만드는 사람의 핵심 역할입니다.

---
## 2. 되먹임 — 실패를 모델에게 알려준다

지금까지는 실패하면 그냥 예외를 던졌습니다. 이제 다르게 합니다.

```
1차 시도   →  {"test": "티검정"}
              ↓ 검증 실패
되먹임     →  "'test' 의 값 '티검정' 는 허용 목록에 없습니다.
               가능한 값: [...]  고쳐서 JSON만 다시 출력하세요."
              ↓
2차 시도   →  {"test": "independent-t-test"}   ✓
```

**실패한 응답과 오류를 대화에 남긴 채로** 다시 요구하는 것이 핵심입니다.
모델은 자기가 뭐라고 했는지, 왜 틀렸는지를 보면서 고칩니다.

In [ ]:
# 되먹임 루프를 직접 구현해 봅니다 (structured.ask_json 의 축약판)
from structured import parse_json, build_json_prompt, JSONParseError

def 내_ask_json(prompt, schema, n_retry=3, verbose=True, **kw):
    kw.setdefault("temperature", 0.0)
    messages = [{"role": "user", "content": build_json_prompt(prompt, schema)}]

    for 시도 in range(1, n_retry + 1):
        원문 = llm.chat(messages, **kw)
        if verbose:
            print(f"[{시도}회] {원문[:90]!r}")

        try:
            data = parse_json(원문)
        except JSONParseError as exc:
            피드백 = f"JSON으로 읽을 수 없었습니다. 오류: {exc}\n설명 없이 JSON만 다시 출력하세요."
        else:
            문제 = validate(data, schema)
            if not 문제:
                if verbose:
                    print(f"      -> 성공 ({시도}회 만에)")
                return data
            피드백 = ("내용에 문제가 있습니다:\n"
                     + "\n".join(f"- {p}" for p in 문제)
                     + "\n고쳐서 JSON만 다시 출력하세요.")

        if verbose:
            print(f"      -> 실패, 되먹임")

        messages.append({"role": "assistant", "content": 원문})
        messages.append({"role": "user", "content": 피드백})

    raise RuntimeError(f"{n_retry}회 모두 실패")


질문 = ("성별(남/여)에 따른 월소득의 평균 차이를 검정하려고 한다. "
        "표본은 각 집단 50명이고 정규성 가정은 만족한다. 어떤 검정을 써야 하는가?")

결과 = 내_ask_json(질문, 검정_스키마, max_tokens=400)
print()
print(결과)

> **되먹임이 실제로 일어났는지** 위 출력에서 확인하세요.
> 1회에 성공했다면 스키마를 더 까다롭게 만들어 다시 돌려 보십시오.

In [ ]:
# 일부러 어렵게: 모델이 잘 모르는 값만 허용해 되먹임을 유도
어려운_스키마 = {
    "required": ["test", "alpha", "n_groups"],
    "types":    {"test": str, "alpha": float, "n_groups": int},
    "allowed":  {"test": ["INDEPENDENT_T_TEST", "PAIRED_T_TEST", "ONE_WAY_ANOVA"]},
    "range":    {"alpha": (0.001, 0.1), "n_groups": (2, 10)},
}

try:
    결과 = 내_ask_json(질문, 어려운_스키마, n_retry=4, max_tokens=400)
    print()
    print("최종:", 결과)
except RuntimeError as exc:
    print()
    print("끝내 실패:", exc)

---
## 3. 공식 구현 `structured.ask_json()`

같은 아이디어에 몇 가지가 더 들어 있습니다.

| 추가 기능 | 왜 |
|---|---|
| `example=` (few-shot) | 1번 노트북에서 확인한 대로 가장 효과가 큼 |
| `system=` 지원 | 역할 고정 |
| `llm=` 주입 | 다른 서버로 바꿔 가며 실험 가능 |
| 실패 시 원인별 예외 | `JSONParseError` vs `SchemaError` 구분 |

In [ ]:
from structured import ask_json

예시 = {"test": "independent-t-test", "alpha": 0.05, "reason": "두 독립 표본의 평균 비교"}

결과 = ask_json(
    질문,
    system="너는 통계 분석 계획을 세우는 도구다.",
    schema=검정_스키마,
    example=예시,
    llm=llm,
    verbose=True,
    max_tokens=400,
)
print()
print(결과)

---
## 4. 단계별 성공률 (ablation)

각 방어층이 실제로 얼마나 기여하는지 측정합니다.
**하나씩 얹으면서** 성공률을 재는 방식입니다.

In [ ]:
import json
import pandas as pd
from structured import parse_json, build_json_prompt

질문들 = [
    "성별에 따른 월소득 평균 차이 검정 (각 50명, 정규성 만족)",
    "치료 전후 혈압 차이 검정 (동일 환자 30명)",
    "세 가지 교육방법에 따른 점수 차이 검정 (각 25명)",
    "흡연여부와 질병유무의 연관성 검정 (2x2 분할표)",
    "키와 몸무게의 선형 관계 검정 (100명)",
]
N_반복 = 4          # 질문 5개 x 4회 = 20 시도/조건

def 층1_그냥(q):
    raw = llm.ask(q + " JSON으로 답하세요.", temperature=0.0, max_tokens=400)
    return json.loads(raw)

def 층2_형식지시(q):
    raw = llm.ask(build_json_prompt(q, 검정_스키마), temperature=0.0, max_tokens=400)
    return json.loads(raw)

def 층3_파서(q):
    raw = llm.ask(build_json_prompt(q, 검정_스키마), temperature=0.0, max_tokens=400)
    return parse_json(raw)

def 층4_검증(q):
    data = 층3_파서(q)
    문제 = validate(data, 검정_스키마)
    if 문제:
        raise ValueError("; ".join(문제))
    return data

def 층5_재시도(q):
    return ask_json(q, schema=검정_스키마, example=예시,
                    llm=llm, n_retry=3, max_tokens=400)

조건들 = {
    "1. 그냥 JSON 요청":      층1_그냥,
    "2. + 형식 지시":         층2_형식지시,
    "3. + 파서":             층3_파서,
    "4. + 스키마 검증":       층4_검증,
    "5. + 되먹임 재시도":     층5_재시도,
}
print("측정 시작 (조건 5 x 시도 20 = 100회 호출, 몇 분 걸립니다)")

In [ ]:
결과기록 = []
for 이름, fn in 조건들.items():
    print(f"{이름:22s}", end=" ")
    성공 = 0
    for q in 질문들:
        for _ in range(N_반복):
            try:
                fn(q)
                성공 += 1
            except Exception:
                pass
        print(".", end="", flush=True)
    n = len(질문들) * N_반복
    결과기록.append({"조건": 이름, "성공": 성공, "n": n})
    print(f"  {성공}/{n}")

abl = pd.DataFrame(결과기록)

In [ ]:
from statsmodels.stats.proportion import proportion_confint

abl["성공률"] = abl["성공"] / abl["n"]
ci = [proportion_confint(r.성공, r.n, method="wilson") for r in abl.itertuples()]
abl["95% CI"] = [f"[{lo:.2f}, {hi:.2f}]" for lo, hi in ci]
abl[["조건", "성공", "성공률", "95% CI"]]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(9, 4))
y = np.arange(len(abl))
lo = abl["성공률"] - np.array([c[0] for c in ci])
hi = np.array([c[1] for c in ci]) - abl["성공률"]

ax.barh(y, abl["성공률"], xerr=[lo, hi], capsize=4, color="#4C72B0")
ax.set_yticks(y)
ax.set_yticklabels([f"L{i+1}" for i in range(len(abl))])
ax.set_xlim(0, 1.05)
ax.set_xlabel("success rate (with 95% Wilson CI)")
ax.set_title("Defense layers ablation")
ax.invert_yaxis()
for i, v in enumerate(abl["성공률"]):
    ax.text(v + 0.02, i, f"{v:.0%}", va="center")
plt.tight_layout()
plt.show()

print()
for i, r in enumerate(abl.itertuples(), 1):
    print(f"L{i} = {r.조건}")

### 무엇을 확인했나

**1. 각 층이 실제로 기여합니다.** 특히 파서와 재시도의 기여가 큽니다.

**2. 그럼에도 100%는 되지 않습니다.**

두 번째가 더 중요합니다.

> **에이전트는 완벽해지지 않습니다.**
> 우리가 할 수 있는 것은 실패율을 낮추고,
> 실패했을 때 **조용히 틀린 답을 내놓는 대신 명확하게 실패**하도록 만드는 것입니다.
>
> 통계 분석 도구라면 특히 그렇습니다.
> 틀린 p값을 자신 있게 말하는 것보다 "판단할 수 없다"고 하는 편이 훨씬 낫습니다.

---
## 5. 이 구조를 기억하십시오

```
        ┌──────────────────────────────┐
        │  행동 (모델에게 요청)          │
        │        ↓                     │
        │  관찰 (파싱 · 검증)            │
        │        ↓                     │
        │  실패면 → 피드백을 만들어      │
        │           대화에 붙이고 재시도  │
        └──────────────────────────────┘
```

| 주차 | 무엇을 요청 | 무엇을 관찰 | 피드백 |
|---|---|---|---|
| **3주 (오늘)** | JSON 출력 | 파싱·스키마 검증 | 검증 오류 메시지 |
| 6주 | 도구 호출 | 도구 실행 결과 | 실행 오류 |
| 7주 | 분석 코드 | 코드 실행 결과 | `Traceback` |
| 12주 | 분석 초안 | Critic 에이전트의 검토 | 반려 사유 |

**전부 같은 구조입니다.** 오늘 배운 것은 JSON 요령이 아니라 자기수정 루프입니다.

---
## 연습문제

**문제 1.** `ask_json()` 에서 `example=` 을 뺐을 때와 넣었을 때의 성공률을
20회씩 비교하고, 두 비율의 차이를 검정하시오.

**문제 2.** `n_retry` 를 1, 2, 3, 5 로 바꿔 가며 성공률과 **평균 호출 횟수**를
함께 재시오. 비용 대비 효과가 가장 좋은 값은?

**문제 3.** 검정 선택 스키마에 규칙을 추가하시오:
`"n_groups"` 가 3 이상이면 `"test"` 는 `one-way-anova` 여야 한다.
지금의 `validate()` 로는 표현할 수 없다. 어떻게 구현하겠는가?

**문제 4.** 자신의 연구 주제에서 LLM에게 시킬 만한 판단을 하나 고르고,
스키마를 설계해 `ask_json()` 으로 10회 돌려 성공률을 재시오.

In [ ]:
# 여기에 직접 작성해 보세요

<details>
<summary><b>해설 보기</b></summary>

**문제 2.** `n_retry` 를 늘리면 성공률은 오르지만 수확이 급격히 체감합니다.
대개 **3회면 충분**합니다. 3회를 넘겨 실패한다면 그것은 재시도로 풀 문제가
아니라 프롬프트나 스키마 설계를 고칠 문제입니다.
평균 호출 횟수도 함께 재야 "성공률 5%p 올리려고 호출을 2배 쓴다"는 판단을
할 수 있습니다.

**문제 3.** 지금의 `validate()` 는 **키별 독립 규칙**만 검사합니다.
키 사이의 관계(조건부 제약)는 표현할 수 없습니다. 두 가지 방법이 있습니다.

*방법 A — 검증 함수를 스키마에 넣는다*
```python
def 규칙_집단수와검정(d):
    if d.get("n_groups", 0) >= 3 and d.get("test") != "one-way-anova":
        return "집단이 3개 이상이면 one-way-anova 여야 합니다."
    return None

schema["custom"] = [규칙_집단수와검정]
```
그리고 `validate()` 끝에 `custom` 규칙들을 돌리는 코드를 추가합니다.

*방법 B — pydantic 의 `model_validator` 를 쓴다*

실무에서는 B가 낫지만, 우리 수업은 **검증이 무엇을 하는 일인지 보이게 하려고**
A 방식으로 직접 만들었습니다. 9주차에 검정 선택 규칙을 본격적으로 다룰 때
이 확장을 실제로 하게 됩니다.

</details>

---
## 3주차 정리

1. 프롬프트는 **함수 명세**다. 출력 형식을 반드시 명시한다
2. **파싱 → 검증 → 되먹임 재시도**, 세 겹으로 막는다
3. 도메인 지식(허용 값, 유효 범위)을 **검증 규칙으로 코드에 심는다**
4. 관대한 파서는 위험하다. **조용히 틀리느니 명확히 실패**
5. 되먹임 루프는 앞으로 모든 주차에서 반복된다

## 내 저장소에 저장하기

`structured.py` 은(는) 이 노트북 위쪽 **준비 셀이 이미 만들어 두었습니다.**
따로 내려받을 것이 없습니다. 자기 저장소로 옮기려면 clone 한 폴더에 복사하세요.

The file(s) above were already written by the setup cells near the top of this
notebook — nothing to download. To keep them in your own repository, copy them
into your cloned folder.

```bash
# repo = 자기 저장소를 clone 한 폴더 / your cloned repository folder
!cp structured.py repo/src/
```


## 다음 주 예고

오늘 만든 `ask_json()` 은 실패할 때마다 대화에 메시지를 두 개씩 덧붙입니다.
재시도를 세 번 하면 대화가 일곱 개로 늘어납니다.
에이전트가 열 단계를 돌면 어떻게 될까요?

다음 주에는 **컨텍스트 윈도우의 한계, 토큰 예산, 대화 압축**을 다룹니다.